
# 🔐 Nemo Safe Synthesizer Tutorial: The Basics

#### What you'll learn

In this notebook, we'll explore the fundamentals of the NeMo Safe Synthesizer: PII replacement, training on a sample dataset, generating synthetic data, and evaluating quality and privacy.

This library supports numeric, categorical, and text fields within the training data and generates realistic synthetic data that mirrors the structure of your data. A full run takes ~15 minutes on an A100; an H100 is faster.

### 🖥️ Prerequisites

This notebook is intended to run on a **GPU**. We recommend an **H100**; minimum **A100**.




### ⚡ Colab Setup

Run the cell below to install Nemo Safe Synthesizer (engine and CUDA 12.8) and the `datasets` library for the sample dataset.

In [ ]:
%%capture
#TODO: Replace the internal NVIDIA Artifactory endpoint with the public PyPI release when available.
!uv pip install nemo-safe-synthesizer[engine,cu128] --extra-index-url "https://urm.nvidia.com/artifactory/api/pypi/nv-shared-pypi-local/simple"
!uv pip install datasets



### 🔑 Set the NIM API key and configure column classification

Setting `NVIDIA_API_KEY` is optional but strongly recommended.

NeMo Safe Synthesizer uses an LLM‑based column classifier to automatically infer column types and improve PII detection accuracy. To enable this feature, you must set both `NIM_ENDPOINT_URL` and `NVIDIA_API_KEY`. You can obtain an API key from [build.nvidia.com](https://build.nvidia.com/settings/api-keys)


In [ ]:
import os
import getpass

# Set the NIM endpoint URL
os.environ["NIM_ENDPOINT_URL"] = "https://integrate.api.nvidia.com/v1"
print("NIM_ENDPOINT_URL is set.")

# Setting NVIDIA_API_KEY is optional but strongly recommended for PII replacement.
if "NVIDIA_API_KEY" not in os.environ:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Paste NIM API key (or press Enter to skip): ")
if os.environ.get("NVIDIA_API_KEY"):
    print("NVIDIA_API_KEY is set")
else:
    print(
        "NVIDIA_API_KEY is not set. "
        "We strongly recommend setting a key."
    )

### 📥 Load and preview sample dataset

Load a tabular dataset—in this example, the [clinc_oos](https://huggingface.co/datasets/clinc/clinc_oos) from Huggingface—and preview the first few rows. NeMo Safe Synthesizer will use this DataFrame as its training data.

This dataset includes a text column and a categorical intent label, making it a good demonstration of multi-type synthesis."

In [ ]:
from datasets import load_dataset

dataset = load_dataset("clinc/clinc_oos", "small")
df = dataset["train"].to_pandas()
df.head()




###  ⚙️ Create and run Safe Synthesizer job

Create the Safe Synthesizer builder and attach your DataFrame. 
Run the pipeline with `run()`, which performs data processing, PII replacement, training, generation, and evaluation in a single call. Results are available on `builder.results`.

 Please refer to the [configuration docs](https://github.com/NVIDIA-NeMo/Safe-Synthesizer/blob/main/docs/user-guide/configuration.md) for the full list of options.



In [ ]:
from nemo_safe_synthesizer.sdk.library_builder import SafeSynthesizer

builder = SafeSynthesizer().with_data_source(df).with_replace_pii()

builder.run()
results = builder.results

### 📤 Retrieve synthetic data

Inspect the generated synthetic data including row count and preview of the first rows.

In [ ]:
synth = results.synthetic_data
print(f"Number of synthetic rows: {len(synth)}")
synth.head()

### 🛡️ Review evaluation report

The pipeline computes both quality and privacy metrics. The summary includes timing information and overall scores, while the full evaluation report is rendered as an HTML document.

In [ ]:
import json

print("Summary (timing and scores):")
print(json.dumps(results.summary.model_dump(), indent=2))

In [ ]:
# Download the full HTML evaluation report
if results.evaluation_report_html:
    report_path = "evaluation_report.html"
    with open(report_path, "w") as f:
        f.write(results.evaluation_report_html)
    print(f"The HTML evaluation report is saved in {report_path}.")